<a href="https://colab.research.google.com/github/winniewyl/super-bowl-ad-analysis/blob/main/notebooks/data_merge_and_feature_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# SECTION 0: Mount Google Drive
# ================================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ================================================================
# SECTION 1: Load YouTube and Reddit CSVs
# ================================================================
import pandas as pd

gemini_path = '/content/drive/MyDrive/super-bowl-ad-analysis/data/raw/youtube_gemini/video_summaries_sentiment.csv'
reddit_path = '/content/drive/MyDrive/super-bowl-ad-analysis/data/raw/reddit_threads/reddit_superbowl_comments.csv'

gemini_df = pd.read_csv(gemini_path)
reddit_df = pd.read_csv(reddit_path)

print("✅ Loaded YouTube entries:", len(gemini_df))
print("✅ Loaded Reddit comments:", len(reddit_df))

✅ Loaded YouTube entries: 154
✅ Loaded Reddit comments: 5468


In [3]:
# ================================================================
# SECTION 2: Sentiment Analysis and Reddit Aggregation
# ================================================================
from textblob import TextBlob

reddit_df['sentiment_score'] = reddit_df['comment_text'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Aggregate by search_term
reddit_agg = reddit_df.groupby('search_term').agg({
    'sentiment_score': 'mean',
    'comment_score': 'mean',
    'comment_text': 'count'
}).reset_index().rename(columns={
    'sentiment_score': 'avg_reddit_sentiment',
    'comment_score': 'avg_comment_score',
    'comment_text': 'num_comments'
})

print("✅ Aggregated Reddit stats:", reddit_agg.shape)

✅ Aggregated Reddit stats: (78, 4)


In [4]:
# ================================================================
# SECTION 3: Merge YouTube and Reddit Datasets
# ================================================================
# Ensure search_term exists in gemini_df
def build_search_term(row):
    brand = row['brand'] if pd.notna(row['brand']) else ''
    year = str(row['year']) if pd.notna(row['year']) else ''
    return f"{brand} Super Bowl {year} ad".strip()

gemini_df['search_term'] = gemini_df.apply(build_search_term, axis=1)

# Merge
merged_df = pd.merge(gemini_df, reddit_agg, on='search_term', how='inner')
print("✅ Merged dataset shape:", merged_df.shape)
merged_df[['brand', 'year', 'search_term', 'avg_reddit_sentiment', 'avg_comment_score', 'num_comments']].head()

✅ Merged dataset shape: (153, 9)


,brand,year,search_term,avg_reddit_sentiment,avg_comment_score,num_comments
0,State Farm,2024,State Farm Super Bowl 2024 ad,0.028495,212.419355,31
1,Liquid Death,2025,Liquid Death Super Bowl 2025 ad,0.019078,1076.220000,100
2,Liquid Death,2025,Liquid Death Super Bowl 2025 ad,0.019078,1076.220000,100
3,OpenAI,2025,OpenAI Super Bowl 2025 ad,0.129578,274.250000,40
4,OpenAI,2025,OpenAI Super Bowl 2025 ad,0.129578,274.250000,40


In [5]:
# ================================================================
# SECTION 4: Add Success Label and Export Cleaned Data
# ================================================================
# Binary label based on Reddit metrics
merged_df['success'] = ((merged_df['avg_reddit_sentiment'] > 0.1) & (merged_df['avg_comment_score'] > 1.5)).astype(int)

# Save cleaned version
output_path = '/content/drive/MyDrive/super-bowl-ad-analysis/data/clean/merged_ad_data.csv'
merged_df.to_csv(output_path, index=False)
print(f"✅ Saved merged dataset with labels to: {output_path}")

✅ Saved merged dataset with labels to: /content/drive/MyDrive/super-bowl-ad-analysis/data/clean/merged_ad_data.csv


In [6]:
# ================================================================
# SECTION 5: Summary of Success Labels
# ================================================================
successful_ads = merged_df['success'].sum()
total_ads = len(merged_df)
print(f"🎯 {successful_ads} out of {total_ads} ads were labeled as successful.")

# Show class distribution
print("\n🔢 Success label distribution:")
print(merged_df['success'].value_counts())

# Show success rate
success_rate = merged_df['success'].mean()
print(f"\n📊 Success rate: {success_rate:.2%}")

🎯 65 out of 153 ads were labeled as successful.

🔢 Success label distribution:
success
0    88
1    65
Name: count, dtype: int64

📊 Success rate: 42.48%
